In [10]:
import pandas as pd

# 1. Load the two CSV files
# NOTE: Make sure these files are in the same directory as your Jupyter notebook
# or provide the full file path.
file_2022_03_16 = 'coin_gecko_2022-03-16.csv'
file_2022_03_17 = 'coin_gecko_2022-03-17.csv'

df_2022_03_16 = pd.read_csv(file_2022_03_16)
df_2022_03_17 = pd.read_csv(file_2022_03_17)

# 2. Concatenate the DataFrames into a single, comprehensive DataFrame
# We ignore the index to create a new, sequential index for the combined data.
df_combined = pd.concat([df_2022_03_16, df_2022_03_17], ignore_index=True)

print("--- Combined DataFrame Head (First 5 Rows) ---")
print(df_combined.head())

print("\n--- Combined DataFrame Info (Columns and Data Types) ---")
df_combined.info()

# Now, 'df_combined' holds all your historical data for the next steps:
# Data Preprocessing, Feature Engineering, and Model Training.

--- Combined DataFrame Head (First 5 Rows) ---
       coin symbol         price     1h    24h     7d    24h_volume  \
0   Bitcoin    BTC  40859.460000  0.022  0.030  0.055  3.539076e+10   
1  Ethereum    ETH   2744.410000  0.024  0.034  0.065  1.974870e+10   
2    Tether   USDT      1.000000 -0.001 -0.001  0.000  5.793497e+10   
3       BNB    BNB    383.430000  0.018  0.028  0.004  1.395854e+09   
4  USD Coin   USDC      0.999874 -0.001  0.000 -0.000  3.872274e+09   

        mkt_cap        date  
0  7.709915e+11  2022-03-16  
1  3.271044e+11  2022-03-16  
2  7.996516e+10  2022-03-16  
3  6.404382e+10  2022-03-16  
4  5.222214e+10  2022-03-16  

--- Combined DataFrame Info (Columns and Data Types) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   coin        1000 non-null   object 
 1   symbol      1000 non-null   object 
 2   price       10

2. 	 Data Preprocessing: Handle missing values, clean data, and normalize numerical features

In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# Assuming df_combined is already created from your previous cell (Step 1: Data Collection)

# --- 2. Data Cleaning ---

# Convert 'date' to datetime objects
df_combined['date'] = pd.to_datetime(df_combined['date'])

# Remove duplicate rows
df_combined.drop_duplicates(inplace=True)
print(f"Number of rows after removing duplicates: {len(df_combined)}")

# --- 3. Handle Missing Values ---
numerical_cols = ['price', '1h', '24h', '7d', '24h_volume', 'mkt_cap']

# 3a. Impute missing values with the mean of that specific cryptocurrency ('symbol')
# FIX: Assign the result directly back to the column to avoid the FutureWarning
for col in numerical_cols:
    df_combined[col] = df_combined[col].fillna(
        df_combined.groupby('symbol')[col].transform('mean')
    )

# 3b. For any remaining NaNs, fill with the overall column mean
# FIX: Use the warning-free assignment method
df_combined[numerical_cols] = df_combined[numerical_cols].fillna(
    df_combined[numerical_cols].mean()
)

print("\nMissing Value Check (Should be 0 for numerical columns):")
print(df_combined[numerical_cols].isnull().sum())

# --- 4. Normalize Numerical Features ---

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply the scaler to the numerical features
df_combined[numerical_cols] = scaler.fit_transform(df_combined[numerical_cols])

# --- Output ---
print("\n--- DataFrame Head (After Preprocessing and Scaling) ---")
print(df_combined.head())

# Save the preprocessed DataFrame to a new CSV file for the next steps
df_combined.to_csv('cryptocurrency_preprocessed.csv', index=False)

Number of rows after removing duplicates: 1000

Missing Value Check (Should be 0 for numerical columns):
price         0
1h            0
24h           0
7d            0
24h_volume    0
mkt_cap       0
dtype: int64

--- DataFrame Head (After Preprocessing and Scaling) ---
       coin symbol     price        1h       24h        7d  24h_volume  \
0   Bitcoin    BTC  8.773501  0.457270  0.106045  0.137326   12.723458   
1  Ethereum    ETH  0.455727  0.531819  0.173767  0.181043    7.053620   
2    Tether   USDT -0.142963 -0.400045 -0.418798 -0.103115   20.895139   
3       BNB    BNB -0.059506  0.308171  0.072184 -0.085628    0.401198   
4  USD Coin   USDC -0.142963 -0.400045 -0.401867 -0.103115    1.298835   

     mkt_cap       date  
0  20.180439 2022-03-16  
1   8.504979 2022-03-16  
2   2.004530 2022-03-16  
3   1.585755 2022-03-16  
4   1.274812 2022-03-16  
